In [27]:
import gc
import logging
import math
import random
import time
import warnings
from datetime import datetime
from operator import itemgetter
from pathlib import Path

logger=logging.getLogger(__name__)
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from collections import defaultdict

from tqdm import tqdm
import pickle

In [13]:
def reduce_mem(df):
    starttime=time.time()
    numerics=['int16','int32','int64','float16','float32','float64']
    start_mem=df.memory_usage().sum()/1024**2
    for col in df.columns:
        if col in numerics:
            c_min=df[col].min()
            c_max=df[col].max()
            if pd.isnull(c_min) or pd.isnull(c_max):
                continue
            if str(df[col].types)[:3]=='int':
                if c_min>np.iinfo(np.int8).min and c_max<np.iinfo(np.int8).max:
                    df[col]=df[col].astype(np.int8)
                elif c_min>np.iinfo(np.int16).min and c_max<np.iinfo(np.int16).max:
                    df[col]=df[col].astype(np.int16)
                elif c_min>np.iinfo[np.int32].min and c_max<np.iinfo(np.int32).max:
                    df[col]=df[col].astype(np.int32)
            else:
                if c_min>np.iinfo(np.float16).min and c_max<np.iinfo(np.float16).max:
                    df[col]=df[col].astype(np.float16)
                elif c_min>np.iinfo(np.float32).min and c_max<np.iinfo(np.float32).max:
                    df[col]=df[col].astype(np.float32)
    end_mem=df.memory_usage().sum()/1024**2
    print(f'--Memory usage after optimization: {end_mem:.2f} MB, about {(start_mem-end_mem)/starttime*100}%, time spend {(time.time()-starttime)/60} min')

In [16]:
def get_all_click_sample(sample_nums=10000):
    """
    训练集中采样一部分数据调试
    :param sample_nums:
    :return:
    """
    all_click=pd.read_csv('./data/train_click_log.csv')
    all_user_ids=all_click.user_id.unique()

    sample_user_ids=np.random.choice(all_user_ids,size=sample_nums,replace=False)
    all_click=all_click[all_click['user_id'].isin(sample_user_ids)]

    # 按照多列去重，('user_id','click_article_id','click_timestamp')都一样才视为相同
    all_click=all_click.drop_duplicates(['user_id','click_article_id','click_timestamp'])
    return all_click

def get_all_click_df(offline):
    if offline:
        all_click=pd.read_csv('./data/train_click_log.csv')[:20000]
    else:
        all_click=pd.concat([pd.read_csv('./data/train_click_log.csv')[:10000],pd.read_csv('./data/testA_click_log.csv')[:10000]])
    return all_click

In [30]:
all_click_df=get_all_click_df(offline=False)
all_click_df.head()

,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,199999,160417,1507029570190,4,1,17,1,13,1
1,199999,5408,1507029571478,4,1,17,1,13,1
2,199999,50823,1507029601478,4,1,17,1,13,1
3,199998,157770,1507029532200,4,1,17,1,25,5
4,199998,96613,1507029671831,4,1,17,1,25,5


In [31]:
def get_user_item_time(click_df):
    '''获取用户-文章-点击时间字典 {user1:{item1:time1,item2:time2}}'''
    memo={}
    for user_id,group in click_df.groupby('user_id'):
        group_sorted=group.sort_values('click_timestamp')
        item_time_dict=dict(zip(group['click_article_id'],group['click_timestamp']))
        memo[user_id]=item_time_dict
    return memo

In [32]:
def get_item_topk_click(click_df,k):
    '''获取点击最多的前k个文章'''
    return click_df['click_article_id'].value_counts().index[:k]

In [37]:
tmp=get_user_item_time(all_click_df)
tmp

{196381: {233917: 1507036600306},
 196382: {5408: 1507036793030, 206661: 1507036823030},
 196383: {50823: 1507039157875, 206934: 1507039187875},
 196384: {156624: 1507036261249, 158536: 1507036291249},
 196385: {64329: 1507036351078, 156560: 1507036381078},
 196386: {62599: 1507036220261, 195843: 1507036250261},
 196387: {156624: 1507036223016, 158536: 1507036253016},
 196388: {214800: 1507119006577, 58606: 1507119036577},
 196389: {156560: 1507041936443,
  308156: 1507043103354,
  123365: 1507043203520,
  286161: 1507043233520},
 196390: {336476: 1507036542066, 272143: 1507036572066},
 196391: {64329: 1507036349694, 272143: 1507036402548, 199198: 1507036432548},
 196392: {286161: 1507036285554, 300074: 1507037854664, 300470: 1507037884664},
 196393: {286161: 1507039185835, 283041: 1507039215835},
 196394: {182513: 1507039011330, 129029: 1507039041330},
 196395: {236726: 1507036551897, 157332: 1507036581897},
 196396: {233717: 1507115265310, 293301: 1507136606607, 123909: 1507136636607

In [41]:
def itemcf_sim(df):
    '''计算物品的相似度矩阵'''
    user_item_time_dict =get_user_item_time(df)
    # i2i_sim[i][j]统计i和j共有的受众个数
    i2i_sim=defaultdict(dict)
    # 统计每个物品受众个数
    item_cnt=defaultdict(int)
    for user,item_time_list in tqdm(user_item_time_dict.items()):
        for i,i_click_time in item_time_list.items():
            # 更新
            item_cnt[i]+=1
            for j, j_click_time in item_time_list.items():
                if i!=j:
                    i2i_sim[i].setdefault(j,0)
                    # 加权弱化：用户点击的物品越多，对每对物品的贡献就越小
                    i2i_sim[i][j]+=1/math.log(len(item_time_list)+1)
    i2i_sim_=i2i_sim.copy()

    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_[i][j]=wij/math.sqrt(item_cnt[i]*item_cnt[j])
    # 保存相似度矩阵
    pickle.dump(i2i_sim_, open('./save/itemcf_i2i_sim.pkl', 'wb'))
i2i_sim=itemcf_sim(all_click_df)

100%|██████████| 7721/7721 [00:00<00:00, 108428.46it/s]


In [51]:
def item_based_rec(user_id,user_item_time_dict,i2i_sim,sim_item_topk,recall_item_num,item_topk_click):
    """
    基于物品的协同过滤
    :param user_id: 用户Id
    :param user_item_time_dict: {user1: {item1: time1, item2: time2..}...}
    :param i2i_sim:相似性矩阵
    :param sim_item_topk: 前k个
    :param recall_item_num:要求召回的文章数量
    :param item_topk_click:点击次数最多的文章列表，用于召回补齐
    :return: {item1:score1, item2: score2...}
    """
    # 用户点击过的物品
    user_hist_items=user_item_time_dict[user_id]

    item_rank={}
    for i,click_time in user_hist_items.items():
        # 为点击过的物品找到最相似的sim_item_topk
        for j ,wij in sorted(i2i_sim[i].items(),key=lambda x:x[1],reverse=True)[:sim_item_topk]:
            # 去重
            if j not in user_hist_items:
                item_rank.setdefault(j,0)
                item_rank[j]+=wij
    # 如果结果不满足召回数量，则用热门物品补齐
    if len(item_rank)<recall_item_num:
        for i,item in enumerate(item_topk_click):
            if item not in item_rank:
                item_rank[item]=i-100
                if len(item_rank)==recall_item_num:break
    # 保留得分最高的
    item_rank=sorted(item_rank.items(),key=lambda x:x[1],reverse=True)[:recall_item_num]
    return item_rank

In [52]:
user_recall_items_dict=defaultdict(dict)
user_item_time_dict=get_user_item_time(all_click_df)

# 物品相似度矩阵
i2i_sim=pickle.load(open('./save/itemcf_i2i_sim.pkl','rb'))

# 相似文章数量
sim_item_topk=10
# 找回个数
recall_item_num=10
# 热门物品个数
item_topk_click=get_item_topk_click(all_click_df,k=50)

# 为每个用户推荐
for user in tqdm(all_click_df['user_id'].unique()):
    user_recall_items_dict[user]=item_based_rec(user,user_item_time_dict,i2i_sim,sim_item_topk,recall_item_num,item_topk_click)


 14%|█▍        | 1083/7721 [00:00<00:01, 5230.04it/s]

[(107301, 0.1786889993417549), (50864, 0.15074178380379075), (160974, 0.14414482050765215), (50383, 0.13514557010693226), (158536, 0.11041310336685785), (161160, 0.10840791429656242), (207469, 0.09464216817386983), (162655, 0.0938444249093069), (156624, 0.08042054203791298), (50619, 0.07762341432436974)]
[(119044, 0.1600208841973003), (273394, 0.1600208841973003), (206159, 0.1600208841973003), (69433, 0.1600208841973003), (206606, 0.1600208841973003), (161227, 0.1600208841973003), (161363, 0.1600208841973003), (225463, 0.022050351725411414), (300470, 0.01476025681000086), (160974, 0.008794040961419797)]
[(225499, 0.25365892483943775), (292708, 0.15610473950771406), (70673, 0.12371007915228557), (237055, 0.12371007915228557), (225514, 0.11038271988126416), (342917, 0.11038271988126416), (166430, 0.07913401120276446), (224857, 0.071424047500042), (330745, 0.06896974169838807), (166445, 0.06152132588197724)]
[(160974, 0.24817664391348115), (31267, 0.1316994362492872), (97661, 0.1134397198

 37%|███▋      | 2866/7721 [00:00<00:00, 5866.86it/s]

[(272143, 0.6280598980756625), (175040, 0.4260683003395364), (198659, 0.2647855906695419), (156624, 0.2369407160131527), (182394, 0.15599021573134333), (160417, 0.1532838659616381), (336476, 0.1251411253450897), (157332, 0.11168702164090244), (348111, 0.10988935039437231), (158536, 0.08967731152361727)]
[(199198, 0.2742467191433369), (118773, 0.21967507280760062), (118878, 0.21967507280760062), (118948, 0.21967507280760062), (118627, 0.21967507280760062), (64329, 0.19735844511100942), (198659, 0.16417387594471664), (175040, 0.14832189638870905), (118726, 0.12682946241971887), (96663, 0.11377990332835466)]
[(199198, 0.4587752611506537), (64329, 0.43889820344322905), (336476, 0.17635741017832535), (198659, 0.16417387594471664), (348111, 0.13272622991294591), (324823, 0.12228561758866816), (156560, 0.1042388584525598), (156624, 0.08526492529654099), (284474, 0.07075792228133822), (328760, 0.05581034781872852)]
[(199198, 0.49483930091243444), (175040, 0.38986165472092865), (198659, 0.20908

 45%|████▍     | 3454/7721 [00:00<00:00, 5226.41it/s]

...

[(160974, 0.3852629720949667), (158536, 0.17225199152517098), (162655, 0.14953685037965625), (202557, 0.14046236273292292), (225463, 0.10758166706668083), (16129, 0.08257752929398565), (156624, 0.08042054203791298), (272660, 0.07848293595695409), (158082, 0.0706363249823161), (59758, 0.06917132404063263)]
[(129086, 0.2637320450404263), (160974, 0.24111815158731453), (78346, 0.19648335830400424), (129687, 0.17649007662455862), (84493, 0.17649007662455862), (141655, 0.15207340994129584), (141517, 0.15207340994129584), (129308, 0.14392145858854952), (202557, 0.14046236273292292), (129238, 0.13893471504706753)]
[(105121, 0.17129944745658357), (161169, 0.1093638945573481), (119044, 0.07543456823158552), (273394, 0.07543456823158552), (206159, 0.07543456823158552), (69433, 0.07543456823158552), (206606, 0.07543456823158552), (161227, 0.07543456823158552), (161363, 0.07543456823158552), (160974, -100)]
[(16129, 0.4176288324229547), (225463, 0.33310014714445485), (158536, 0.22092988066584912),

 90%|████████▉ | 6932/7721 [00:01<00:00, 4850.07it/s]

[(16129, 0.24817664391348115), (300470, 0.24111815158731453), (162300, 0.21585021382334774), (202557, 0.1733634389310479), (225463, 0.17099839583552265), (158536, 0.159090992507536), (160417, 0.14414482050765215), (59758, 0.12538033733281437), (156808, 0.09421911878204836), (158082, 0.08998931365296603)]
[(300470, 0.4014764348351241), (16129, 0.35730069788632535), (162300, 0.32110275999644333), (158536, 0.26950409587439383), (202557, 0.22788352317329924), (59758, 0.18833970444528786), (158082, 0.16062563863528212), (156808, 0.09421911878204836), (162655, 0.0938444249093069), (156624, 0.08042054203791298)]
[(202557, -91), (162655, -92), (158536, -93), (199198, -94), (160417, -95), (16129, -96), (156624, -97), (300470, -98), (272143, -99), (160974, -100)]
[(16129, 0.35730069788632535), (300470, 0.34869981865399535), (162300, 0.2600269554723311), (202557, 0.22788352317329924), (59758, 0.18833970444528786), (158536, 0.159090992507536), (160417, 0.14414482050765215), (156808, 0.094219118782

100%|██████████| 7721/7721 [00:01<00:00, 4856.13it/s]

[(160974, 0.25189598096294374), (59758, 0.19102988849461972), (300470, 0.18606460302363492), (16129, 0.16864658898715576), (284844, 0.11097671047150096), (313504, 0.10798741594776293), (96812, 0.066209620404158), (236613, 0.06089593473724781), (273464, 0.05853490036423154), (202557, 0.054520084242251354)]
[(111031, 0.26276343125998064), (352614, 0.1816905013650884), (352461, 0.1816905013650884), (160974, 0.17334377498802903), (202528, 0.14575492688072578), (283933, 0.1367782296577098), (59310, 0.11801902116367535), (57881, 0.11801902116367535), (68922, 0.11801902116367535), (58867, 0.11801902116367535)]
[(160974, 0.3852629720949667), (158536, 0.17225199152517098), (162655, 0.14953685037965625), (202557, 0.14046236273292292), (225463, 0.10758166706668083), (16129, 0.08257752929398565), (156624, 0.08042054203791298), (272660, 0.07848293595695409), (158082, 0.0706363249823161), (59758, 0.06917132404063263)]
[(160974, 0.24817664391348115), (225463, 0.1091240539728442), (202557, 0.086874659

In [55]:
user_item_score_list=[]
# 转换为DataFrame查看
for user,memo in user_recall_items_dict.items():
    for item,score in memo:
        user_item_score_list.append([user,item,score])
recall_df=pd.DataFrame(user_item_score_list,columns=['user_id','click_article_id','pred_score'])
recall_df.head()

,user_id,click_article_id,pred_score
0,199999,107301,0.178689
1,199999,50864,0.150742
2,199999,160974,0.144145
3,199999,50383,0.135146
4,199999,158536,0.110413


In [ ]:
def submit(user_recall_items_dict,model_name=None):
    '''生成提交文件'''
    ans=[]
    for user,memo in user_recall_items_dict.items():
        tmp=[]
        # 按照得分降序排列
        for item,score in sorted(memo,key=lambda x:x[1],reverse=True):
            tmp.append(item)
        ans.append([[user]+tmp[:]])
    ans=pd.DataFrame(ans,columns=['user_id', 'article_1', 'article_2','article_3', 'article_4', 'article_5'])
    ans.to_csv('./save/submission.csv',index=False,header=True)